## Find new colin events - split corp nums into batches

In [ ]:
import re
from pathlib import Path
from math import ceil

# ====== CONFIG ======
# In SQL, Oracle type constructors are limited to 999 parameters.
# So keep this <= 999 to avoid ORA-00939 in sys.odcivarchar2list(...).
BATCH_SIZE = 999

# How many corp nums to print per *line* inside sys.odcivarchar2list(...)
# (this is just formatting; set 500-999 as you requested)
VALUES_PER_LINE = 999

CUTOFF = "TIMESTAMP '2026-03-09 23:00:00'"  # or ":cutoff_ts"
OUTPUT_FILE = "generated/check_events.sql"

IDENTIFIERS = r"""
'1111111', '2222222', '3333333'
-- paste your full list here exactly as provided (single quoted, comma-separated)
"""
# ====================

def parse_ids(s: str) -> list[str]:
    # Extract values inside single quotes; preserves leading zeros like '0766566'
    ids = re.findall(r"'([^']*)'", s)

    # De-dupe while preserving order (optional; remove if you want duplicates preserved)
    seen = set()
    out: list[str] = []
    for x in ids:
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out

def format_as_odcivarchar2list_args(items: list[str], per_line: int) -> str:
    """
    Returns a multi-line string of quoted items, comma-separated, with per_line items per line.
    Example (per_line=3):
        'a','b','c',
        'd','e'
    """
    lines: list[str] = []
    for i in range(0, len(items), per_line):
        chunk = items[i:i + per_line]
        lines.append("    " + ",".join(f"'{v}'" for v in chunk))
    return ",\n".join(lines)

def make_batch_cte(cte_name: str, items: list[str]) -> str:
    args_sql = format_as_odcivarchar2list_args(items, VALUES_PER_LINE)
    lines = [
        f"{cte_name} AS (",
        "  SELECT column_value AS corp_num",
        "  FROM TABLE(sys.odcivarchar2list(",
        args_sql,
        "  ))",
        ")",
    ]
    return "\n".join(lines)

corp_nums = parse_ids(IDENTIFIERS)
if not corp_nums:
    raise SystemExit("No corp nums found. Paste the quoted list into IDENTIFIERS.")

n = len(corp_nums)
num_batches = ceil(n / BATCH_SIZE)

batches = [
    corp_nums[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
    for i in range(num_batches)
]

# Build corp_list_001, corp_list_002, ...
cte_blocks = []
batch_names = []
for idx, batch in enumerate(batches, start=1):
    name = f"corp_list_{idx:03d}"
    batch_names.append(name)
    cte_blocks.append(make_batch_cte(name, batch))

cte_blocks_sql = ",\n".join(cte_blocks)

# Build unified corp_list CTE
corp_union_lines = ["corp_list AS ("]
corp_union_lines.append(f"  SELECT corp_num FROM {batch_names[0]}")
for name in batch_names[1:]:
    corp_union_lines.append(f"  UNION ALL SELECT corp_num FROM {name}")
corp_union_lines.append(")")
corp_list_sql = "\n".join(corp_union_lines)

sql = f"""\
WITH
{cte_blocks_sql},
{corp_list_sql},
latest_event AS (
  SELECT e.event_id,
         e.corp_num,
         e.event_typ_cd,
         e.event_timestmp,
         e.trigger_dts,
         ROW_NUMBER() OVER (
           PARTITION BY e.corp_num
           ORDER BY e.event_timestmp DESC
         ) AS rn
  FROM event e
  JOIN corp_list c
    ON c.corp_num = e.corp_num
  WHERE e.event_timestmp > {CUTOFF}
)
SELECT le.EVENT_ID,
       le.corp_num,
       le.event_typ_cd,
       le.event_timestmp,
       le.trigger_dts,
       f.FILING_TYP_CD
FROM latest_event le
left join filing f on le.EVENT_ID = f.EVENT_ID
WHERE rn = 1
ORDER BY event_timestmp DESC;
"""

Path(OUTPUT_FILE).write_text(sql, encoding="utf-8")

print(f"Wrote {OUTPUT_FILE}")
print(f"Total corp nums: {n}")
print(f"Batch size (per odcivarchar2list): {BATCH_SIZE}")
print(f"Values per line (formatting only): {VALUES_PER_LINE}")
print(f"Batches: {num_batches} -> {[len(b) for b in batches]}")
